In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.animation as animation

from scipy import stats
from scipy.spatial import cKDTree
from datetime import datetime
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from matplotlib.lines import Line2D
from matplotlib.patches import Circle, Rectangle
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score, accuracy_score)
from scipy.spatial.distance import cdist

import warnings
warnings.filterwarnings('ignore')

os.makedirs('data', exist_ok=True)
os.makedirs('logs', exist_ok=True)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

PyTorch version: 2.8.0+cu126
CUDA available: True


In [2]:
!pip install torch_geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 16.7 MB/s eta 0:00:00


##HerdAI w Agent-Based Model Simulator

In [3]:
try:
    with open('config.json', 'r') as f:
        config = json.load(f)
except FileNotFoundError:
    print("\nconfig.json not found. Using default parameters.")
    config = {
        "simulation": {
            "grid_size": [50, 50],
            "num_agents": 100,
            "num_steps": 1000,
            "random_seed": 42
        },
        "disease": {
            "base_transmission_prob": 0.15,
            "incubation_period_mean": 48,
            "incubation_period_std": 12,
            "infectious_period_mean": 120,
            "infectious_period_std": 24,
            "transmission_radius": 2.5,
            "initial_infected": 2
        }
    }

np.random.seed(config['simulation']['random_seed'])


config.json not found. Using default parameters.


In [4]:
SUSCEPTIBLE = 0
EXPOSED = 1
INFECTIOUS = 2
RECOVERED = 3
STATE_NAMES = {0: 'SUSCEPTIBLE', 1: 'EXPOSED', 2: 'INFECTIOUS', 3: 'RECOVERED'}

BREEDS = {
    'Holstein': {'temp': 38.6, 'susceptibility': 1.35, 'activity': 95},
    'Angus': {'temp': 38.4, 'susceptibility': 0.70, 'activity': 108},
    'Jersey': {'temp': 38.5, 'susceptibility': 1.15, 'activity': 98},
    'Hereford': {'temp': 38.3, 'susceptibility': 0.80, 'activity': 112}
}

ZONE_PASTURE = 0
ZONE_FEEDING = 1
ZONE_WATER = 2
ZONE_SHELTER = 3

In [5]:
NUM_AGENTS = config['simulation']['num_agents']
NUM_STEPS = config['simulation']['num_steps']
GRID_SIZE = tuple(config['simulation']['grid_size'])

print(f"simulation parameters:")
print(f"\ngrid size: {GRID_SIZE[0]} × {GRID_SIZE[1]} mts")
print(f"population: {NUM_AGENTS} cattle")
print(f"duration: {NUM_STEPS} timesteps ({NUM_STEPS/4:.1f} hrs / {NUM_STEPS/96:.1f} days)")
print(f"transmission rate: {config['disease']['base_transmission_prob']}")
print(f"transmission radius: {config['disease']['transmission_radius']}m")

simulation parameters:

grid size: 50 × 50 mts
population: 100 cattle
duration: 1000 timesteps (250.0 hrs / 10.4 days)
transmission rate: 0.15
transmission radius: 2.5m


In [6]:
print(f"\ninitializing {NUM_AGENTS} heterogeneous agents")

positions = np.random.rand(NUM_AGENTS, 2) * np.array(GRID_SIZE)

health_states = np.zeros(NUM_AGENTS, dtype=int)

exposure_times = np.zeros(NUM_AGENTS, dtype=int)
infection_times = np.full(NUM_AGENTS, -1, dtype=int)
recovery_times = np.full(NUM_AGENTS, -1, dtype=int)

breed_names = list(BREEDS.keys())
breed_weights = [0.35, 0.30, 0.20, 0.15]  #holstein most common
agent_breeds = np.random.choice(breed_names, NUM_AGENTS, p=breed_weights)

agent_ages = np.clip(np.random.gamma(3, 10, NUM_AGENTS).astype(int), 6, 144)

base_temperatures = np.array([BREEDS[b]['temp'] for b in agent_breeds])
susceptibilities = np.array([BREEDS[b]['susceptibility'] for b in agent_breeds])
base_activities = np.array([BREEDS[b]['activity'] for b in agent_breeds])

age_modifiers = np.ones(NUM_AGENTS)
age_modifiers[agent_ages < 12] = 1.4
age_modifiers[agent_ages > 96] = 1.3
susceptibilities *= age_modifiers

social_tendencies = np.random.beta(2, 2, NUM_AGENTS)
movement_speeds = np.random.gamma(2, 0.5, NUM_AGENTS)

initial_infected = min(config['disease']['initial_infected'], NUM_AGENTS)
health_states[:initial_infected] = EXPOSED
exposure_times[:initial_infected] = np.random.randint(0, 24, initial_infected)

print(f"agent initialization complete")
print(f"breeds: {dict(pd.Series(agent_breeds).value_counts())}")
print(f"age range: {agent_ages.min()}-{agent_ages.max()} mths")
print(f"initial infected: {initial_infected}")


initializing 100 heterogeneous agents
agent initialization complete
breeds: {'Holstein': np.int64(32), 'Angus': np.int64(31), 'Jersey': np.int64(20), 'Hereford': np.int64(17)}
age range: 6-80 mths
initial infected: 2


In [7]:
print(f"\ncreating farm environment")

zone_grid = np.zeros(GRID_SIZE, dtype=int)

feeding_stations = [
    (GRID_SIZE[0]//4, GRID_SIZE[1]//4),
    (3*GRID_SIZE[0]//4, GRID_SIZE[1]//4),
    (GRID_SIZE[0]//4, 3*GRID_SIZE[1]//4),
    (3*GRID_SIZE[0]//4, 3*GRID_SIZE[1]//4)
]

for fx, fy in feeding_stations:
    for dx in range(-2, 3):
        for dy in range(-2, 3):
            x, y = fx + dx, fy + dy
            if 0 <= x < GRID_SIZE[0] and 0 <= y < GRID_SIZE[1]:
                zone_grid[x, y] = ZONE_FEEDING

water_troughs = [
    (GRID_SIZE[0]//2, GRID_SIZE[1]//4),
    (GRID_SIZE[0]//2, 3*GRID_SIZE[1]//4)
]

for wx, wy in water_troughs:
    for dx in range(-1, 2):
        for dy in range(-1, 2):
            x, y = wx + dx, wy + dy
            if 0 <= x < GRID_SIZE[0] and 0 <= y < GRID_SIZE[1]:
                zone_grid[x, y] = ZONE_WATER

shelter_x, shelter_y = GRID_SIZE[0]//2, GRID_SIZE[1]//2
for dx in range(-3, 4):
    for dy in range(-3, 4):
        x, y = shelter_x + dx, shelter_y + dy
        if 0 <= x < GRID_SIZE[0] and 0 <= y < GRID_SIZE[1]:
            zone_grid[x, y] = ZONE_SHELTER

print(f"farm environment created")
print(f"feeding stations: {len(feeding_stations)}")
print(f"water troughs: {len(water_troughs)}")


creating farm environment
farm environment created
feeding stations: 4
water troughs: 2


In [8]:
data_log = []
metrics_log = {
    'susceptible': [],
    'exposed': [],
    'infectious': [],
    'recovered': []
}

In [9]:
print(f"\nstarting simulation")
print("=" * 80)

for step in tqdm(range(NUM_STEPS), desc="Simulating", ncols=80):
    for i in range(NUM_AGENTS):
        if health_states[i] == EXPOSED:
            exposure_times[i] += 1

            #stochastic incubation period
            incubation_threshold = max(24, int(np.random.normal(
                config['disease']['incubation_period_mean'],
                config['disease']['incubation_period_std']
            )))

            if exposure_times[i] >= incubation_threshold:
                health_states[i] = INFECTIOUS
                infection_times[i] = step

        elif health_states[i] == INFECTIOUS:
            time_infectious = step - infection_times[i]

            #stochastic recovery period
            recovery_threshold = max(48, int(np.random.normal(
                config['disease']['infectious_period_mean'],
                config['disease']['infectious_period_std']
            )))

            if time_infectious >= recovery_threshold:
                health_states[i] = RECOVERED
                recovery_times[i] = step

        elif health_states[i] == RECOVERED:
            time_recovered = step - recovery_times[i]
            if time_recovered >= 240 and np.random.random() < 0.02:
                health_states[i] = SUSCEPTIBLE
                infection_times[i] = -1
                recovery_times[i] = -1
                exposure_times[i] = 0


    hour_of_day = (step // 4) % 24

    #goal-directed movement (circadian patterns)
    for i in range(NUM_AGENTS):
        #reduce mobility if sick
        mobility_factor = 1.0
        if health_states[i] == INFECTIOUS:
            mobility_factor = 0.3
        elif health_states[i] == EXPOSED:
            mobility_factor = 0.7

        #determine attraction point based on time
        target = None
        if 6 <= hour_of_day <= 9 or 17 <= hour_of_day <= 20:
            #feding times
            if np.random.random() < 0.7:
                target = np.array(feeding_stations[np.random.randint(len(feeding_stations))])
        elif 12 <= hour_of_day <= 14:
            #water breaks
            if np.random.random() < 0.6:
                target = np.array(water_troughs[np.random.randint(len(water_troughs))])

        #move toward target or random walk
        if target is not None:
            direction = target - positions[i]
            distance = np.linalg.norm(direction)
            if distance > 1.5:
                direction = direction / distance
                step_size = min(movement_speeds[i] * mobility_factor, distance * 0.3)
                positions[i] += direction * step_size
        else:
            #random walk (Lévy flight)
            angle = np.random.uniform(0, 2 * np.pi)
            distance = np.random.gamma(2, 0.5) * movement_speeds[i] * mobility_factor
            positions[i] += distance * np.array([np.cos(angle), np.sin(angle)])

        #boundary conditions
        positions[i] = np.clip(positions[i], [0, 0],
                              [GRID_SIZE[0]-1, GRID_SIZE[1]-1])

    tree = cKDTree(positions)

    infectious_indices = np.where(health_states == INFECTIOUS)[0]

    newly_exposed = []

    for inf_idx in infectious_indices:
        neighbor_indices = tree.query_ball_point(
            positions[inf_idx],
            r=config['disease']['transmission_radius']
        )

        for n_idx in neighbor_indices:
            if n_idx == inf_idx:
                continue

            if health_states[n_idx] != SUSCEPTIBLE:
                continue

            distance = np.linalg.norm(positions[inf_idx] - positions[n_idx])

            spatial_factor = np.exp(-distance / config['disease']['transmission_radius'])

            zone = zone_grid[int(positions[n_idx][0]), int(positions[n_idx][1])]
            zone_multipliers = {
                ZONE_PASTURE: 1.0,
                ZONE_FEEDING: 2.5,
                ZONE_WATER: 1.8,
                ZONE_SHELTER: 2.0
            }
            zone_factor = zone_multipliers.get(zone, 1.0)

            susceptibility = susceptibilities[n_idx]

            prob = (config['disease']['base_transmission_prob'] *
                   spatial_factor *
                   zone_factor *
                   susceptibility)

            prob = min(prob, 0.98)

            if np.random.random() < prob:
                if n_idx not in newly_exposed:
                    newly_exposed.append(n_idx)

    for n_idx in newly_exposed:
        health_states[n_idx] = EXPOSED
        exposure_times[n_idx] = 0

    for i in range(NUM_AGENTS):
        base_temp = base_temperatures[i]

        #circadian rhythm (±0.3°C)
        circadian = 0.3 * np.sin(2 * np.pi * hour_of_day / 24)

        #disease effect
        if health_states[i] == EXPOSED:
            #gradual fever onset
            progress = exposure_times[i] / config['disease']['incubation_period_mean']
            disease_temp = 0.3 + 0.4 * progress
        elif health_states[i] == INFECTIOUS:
            #peak fever
            days_infectious = (step - infection_times[i]) / 24
            disease_temp = 1.8 + 0.3 * min(days_infectious, 3)
        elif health_states[i] == RECOVERED:
            #gradual normalization
            days_recovered = (step - recovery_times[i]) / 24
            disease_temp = max(0, 0.6 - 0.3 * days_recovered)
        else:
            disease_temp = 0.0

        #sensor noise
        noise = np.random.normal(0, 0.12)

        temperature = base_temp + circadian + disease_temp + noise
        temperature = np.clip(temperature, 36.5, 42.0)

        base_activity = base_activities[i]

        if 22 <= hour_of_day or hour_of_day <= 5:
            time_factor = 0.25
        elif 6 <= hour_of_day <= 9 or 17 <= hour_of_day <= 20:
            time_factor = 1.3
        else:
            time_factor = 1.0

        #disease effect
        if health_states[i] == INFECTIOUS:
            disease_activity = 0.35
        elif health_states[i] == EXPOSED:
            disease_activity = 0.85
        elif health_states[i] == RECOVERED:
            days_recovered = (step - recovery_times[i]) / 24
            disease_activity = 0.6 + 0.15 * min(days_recovered, 3)
        else:
            disease_activity = 1.0

        #sensor noise
        noise = np.random.normal(0, 4.0)

        activity = base_activity * time_factor * disease_activity + noise
        activity = np.clip(activity, 0, 200)

        if health_states[i] == INFECTIOUS:
            cough_count = np.random.poisson(5)
            sneeze_count = np.random.poisson(2)
            heavy_breathing = np.random.poisson(8)
        elif health_states[i] == EXPOSED:
            cough_count = np.random.poisson(0.8)
            sneeze_count = 0
            heavy_breathing = np.random.poisson(2)
        elif health_states[i] == RECOVERED:
            cough_count = np.random.poisson(1.2)
            sneeze_count = 0
            heavy_breathing = np.random.poisson(1)
        else:
            cough_count = 1 if np.random.random() < 0.02 else 0
            sneeze_count = 0
            heavy_breathing = 0

        zone = zone_grid[int(positions[i][0]), int(positions[i][1])]
        zone_names = {0: 'PASTURE', 1: 'FEEDING_STATION', 2: 'WATER_TROUGH', 3: 'SHELTER'}

        data_log.append({
            'timestep': step,
            'timestamp_minutes': step * 15,
            'hour_of_day': hour_of_day,
            'agent_id': i,

            'breed': agent_breeds[i],
            'age_months': int(agent_ages[i]),

            'position_x': round(float(positions[i][0]), 2),
            'position_y': round(float(positions[i][1]), 2),
            'zone_type': zone_names[zone],

            'health_state': STATE_NAMES[health_states[i]],
            'health_state_code': int(health_states[i]),
            'days_since_exposure': round(exposure_times[i] / 24, 2),

            'body_temperature': round(temperature, 2),

            'activity_level': round(activity, 1),

            'cough_count': int(cough_count),
            'sneeze_count': int(sneeze_count),
            'heavy_breathing_count': int(heavy_breathing),

            'susceptibility': round(float(susceptibilities[i]), 2),
            'social_tendency': round(float(social_tendencies[i]), 2)
        })

    metrics_log['susceptible'].append(int(np.sum(health_states == SUSCEPTIBLE)))
    metrics_log['exposed'].append(int(np.sum(health_states == EXPOSED)))
    metrics_log['infectious'].append(int(np.sum(health_states == INFECTIOUS)))
    metrics_log['recovered'].append(int(np.sum(health_states == RECOVERED)))


starting simulation


Simulating: 100%|███████████████████████████| 1000/1000 [00:24<00:00, 40.61it/s]


In [10]:
print("savin dataset")

df = pd.DataFrame(data_log)
output_path = 'data/digital_herd_health_dataset.csv'
df.to_csv(output_path, index=False)

print(f"\ndataset saved: {output_path}")
print(f"total records: {len(df):,}")
print(f"shape: {df.shape}")
print(f"file size: {os.path.getsize(output_path) / 1024 / 1024:.2f} MB")

#save metrics
metrics_df = pd.DataFrame(metrics_log)
metrics_df['timestep'] = range(NUM_STEPS)
metrics_df.to_csv('data/epidemic_metrics.csv', index=False)
print(f"metrics saved: data/epidemic_metrics.csv")

#save metadata
metadata = {
    'simulation_date': datetime.now().isoformat(),
    'config': config,
    'num_records': len(df),
    'num_agents': NUM_AGENTS,
    'num_timesteps': NUM_STEPS,
    'features': list(df.columns),
    'breed_distribution': df.groupby('breed')['agent_id'].nunique().to_dict()
}

with open('data/simulation_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)
print(f"metadata saved: data/simulation_metadata.json")

savin dataset

dataset saved: data/digital_herd_health_dataset.csv
total records: 100,000
shape: (100000, 19)
file size: 8.55 MB
metrics saved: data/epidemic_metrics.csv
metadata saved: data/simulation_metadata.json


In [11]:
print("simulation summary")

print(f"\ndataset preview:")
print(df.head(10).to_string(index=False))

#final health states
print(f"\nfinal health states:")
final_states = df[df['timestep'] == NUM_STEPS-1]['health_state'].value_counts()
for state, count in final_states.items():
    pct = count / NUM_AGENTS * 100
    print(f"   {state:12s}: {count:3d} agents ({pct:5.1f}%)")

#attack rate
attack_rate = ((NUM_AGENTS - final_states.get('SUSCEPTIBLE', 0)) / NUM_AGENTS) * 100
print(f"\nepidemiological metrics:")
print(f"attack rate: {attack_rate:.1f}%")
print(f"peak infectious: {max(metrics_log['infectious'])} agents")
print(f"peak timestep: {np.argmax(metrics_log['infectious'])}")

#data quality
print(f"\ndata quality:")
print(f"missing values: {df.isnull().sum().sum()}")
print(f"duplicates: {df.duplicated().sum()}")
print(f"temporal completeness: 100%")

simulation summary

dataset preview:
 timestep  timestamp_minutes  hour_of_day  agent_id    breed  age_months  position_x  position_y       zone_type health_state  health_state_code  days_since_exposure  body_temperature  activity_level  cough_count  sneeze_count  heavy_breathing_count  susceptibility  social_tendency
        0                  0            0         0    Angus          27       18.02       48.07         PASTURE      EXPOSED                  1                 0.88             38.88            24.9            0             0                      3            0.70             0.59
        0                  0            0         1 Holstein          17       36.89       31.17         PASTURE      EXPOSED                  1                 0.62             38.87            17.3            1             0                      1            1.35             0.87
        0                  0            0         2 Holstein           6        6.52        5.99         PASTURE  

In [12]:
print("HerdAI w data analysis & visuals")

os.makedirs('figures', exist_ok=True)
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

print("\nloading dataset")
df = pd.read_csv('data/digital_herd_health_dataset.csv')
print(f"loaded {len(df):,} records")
print(f"shape: {df.shape}")
print(f"time range: {df['timestep'].min()} - {df['timestep'].max()}")

HerdAI w data analysis & visuals

loading dataset
loaded 100,000 records
shape: (100000, 19)
time range: 0 - 999


In [13]:
print("\n[1/8]creating epidemic curves")

state_counts = df.groupby(['timestep', 'health_state']).size().unstack(fill_value=0)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('HerdAI(SEIRS Model)',
             fontsize=18, fontweight='bold')

# Subplot A: SEIR curves
ax = axes[0, 0]
for state in state_counts.columns:
    ax.plot(state_counts.index, state_counts[state],
           label=state, linewidth=2.5, marker='o', markersize=2, alpha=0.8)
ax.set_xlabel('timestep (15-min intervals)', fontsize=12)
ax.set_ylabel('no of animals', fontsize=12)
ax.set_title('a) SEIRS model dynamics', fontsize=14, fontweight='bold')
ax.legend(loc='best', fontsize=11, framealpha=0.9)
ax.grid(True, alpha=0.3)

# Subplot B: Prevalence (%)
ax = axes[0, 1]
total = state_counts.sum(axis=1)
prevalence = state_counts.div(total, axis=0) * 100
for state in prevalence.columns:
    ax.plot(prevalence.index, prevalence[state], label=state, linewidth=2.5, alpha=0.8)
ax.set_xlabel('timestep', fontsize=12)
ax.set_ylabel('prevalence (%)', fontsize=12)
ax.set_title('b)disease prevalence over time', fontsize=14, fontweight='bold')
ax.legend(loc='best', fontsize=11)
ax.grid(True, alpha=0.3)

# Subplot C: Cumulative attack rate
ax = axes[1, 0]
susceptible = state_counts.get('SUSCEPTIBLE', total)
attack_rate = (1 - susceptible / susceptible.iloc[0]) * 100
ax.plot(attack_rate.index, attack_rate, linewidth=4, color='darkred', alpha=0.8)
ax.fill_between(attack_rate.index, attack_rate, alpha=0.3, color='red')
ax.set_xlabel('timestep', fontsize=12)
ax.set_ylabel('cumulative attack rate (%)', fontsize=12)
ax.set_title('c)attack rate progression', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.axhline(y=attack_rate.iloc[-1], color='red', linestyle='--', alpha=0.5,
          label=f'Final: {attack_rate.iloc[-1]:.1f}%')
ax.legend()

# Subplot D: Daily incidence
ax = axes[1, 1]
exposed = state_counts.get('EXPOSED', 0)
infectious = state_counts.get('INFECTIOUS', 0)
incidence = (exposed + infectious).diff().fillna(0)
ax.bar(incidence.index, incidence, alpha=0.7, color='crimson', edgecolor='darkred')
ax.set_xlabel('timestep', fontsize=12)
ax.set_ylabel('new infections', fontsize=12)
ax.set_title('d)daily incidence rate', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
ax.axhline(y=0, color='black', linewidth=0.8)

plt.tight_layout()
plt.savefig('figures/1_epidemic_curves.png', dpi=300, bbox_inches='tight')
print("saved: figures/1_epidemic_curves.png")
plt.close()


[1/8]creating epidemic curves
saved: figures/1_epidemic_curves.png


In [14]:
print("[2/8]creating sensor distributions")

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('multimodal sensor analysis by Health State', fontsize=18, fontweight='bold')

sensors = [
    ('body_temperature', 'body temperature (°C)', axes[0, 0]),
    ('activity_level', 'activity level (score)', axes[0, 1]),
    ('cough_count', 'cough count (per 15min)', axes[0, 2]),
    ('sneeze_count', 'sneeze count', axes[1, 0]),
    ('heavy_breathing_count', 'heavy breathing', axes[1, 1])
]

for col, label, ax in sensors:
    df.boxplot(column=col, by='health_state', ax=ax, patch_artist=True)
    ax.set_title(label, fontsize=12, fontweight='bold')
    ax.set_xlabel('health state', fontsize=10)
    ax.set_ylabel(label, fontsize=10)
    plt.sca(ax)
    plt.xticks(rotation=45, ha='right')

ax = axes[1, 2]
ax.axis('off')
summary_text = "statistical summary:\n\n"
for state in df['health_state'].unique():
    temps = df[df['health_state'] == state]['body_temperature']
    summary_text += f"{state}:\n"
    summary_text += f"temp: {temps.mean():.2f}°C ± {temps.std():.2f}\n"
    summary_text += f"n: {len(temps):,}\n\n"

ax.text(0.1, 0.5, summary_text, fontsize=10, verticalalignment='center',
       bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))

plt.tight_layout()
plt.savefig('figures/2_sensor_distributions.png', dpi=300, bbox_inches='tight')
print("saved: figures/2_sensor_distributions.png")
plt.close()

[2/8]creating sensor distributions
saved: figures/2_sensor_distributions.png


In [15]:
print("[3/8]creating correlation matrix")

numeric_cols = ['body_temperature', 'activity_level', 'cough_count',
               'sneeze_count', 'heavy_breathing_count', 'age_months',
               'days_since_exposure', 'susceptibility']
corr_matrix = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
           center=0, square=True, linewidths=0.5,
           cbar_kws={"shrink": 0.8}, mask=mask, ax=ax,
           vmin=-1, vmax=1)
ax.set_title('feature correlation matrix', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/3_correlation_matrix.png', dpi=300, bbox_inches='tight')
print("saved: figures/3_correlation_matrix.png")
plt.close()

[3/8]creating correlation matrix
saved: figures/3_correlation_matrix.png


In [16]:
print("[4/8]creating spatial distribution")

mid_timestep = df['timestep'].max() // 2
snapshot = df[df['timestep'] == mid_timestep].copy()

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Scatter plot
ax = axes[0]
colors = {'SUSCEPTIBLE': 'green', 'EXPOSED': 'yellow',
         'INFECTIOUS': 'red', 'RECOVERED': 'blue'}

for state, color in colors.items():
    state_data = snapshot[snapshot['health_state'] == state]
    ax.scatter(state_data['position_x'], state_data['position_y'],
              c=color, label=state, alpha=0.7, s=120, edgecolors='black', linewidths=1.5)

ax.set_title(f'a)spatial distribution (timestep {mid_timestep})',
            fontsize=14, fontweight='bold')
ax.set_xlabel('x position (mts)', fontsize=12)
ax.set_ylabel('y position (mts)', fontsize=12)
ax.legend(loc='upper right', fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 50)
ax.set_ylim(0, 50)

# Density heatmap
ax = axes[1]
state_map = {'SUSCEPTIBLE': 0, 'EXPOSED': 1, 'INFECTIOUS': 2, 'RECOVERED': 3}
snapshot['state_code'] = snapshot['health_state'].map(state_map)

grid_size = 50
health_grid = np.zeros((grid_size, grid_size))
count_grid = np.zeros((grid_size, grid_size))

for _, row in snapshot.iterrows():
    x, y = int(row['position_x']), int(row['position_y'])
    if 0 <= x < grid_size and 0 <= y < grid_size:
        health_grid[y, x] += row['state_code']
        count_grid[y, x] += 1

with np.errstate(divide='ignore', invalid='ignore'):
    avg_health = np.divide(health_grid, count_grid)
    avg_health[np.isnan(avg_health)] = -1

im = ax.imshow(avg_health, cmap='RdYlGn_r', interpolation='gaussian', origin='lower')
ax.set_title('b)disease status heatmap', fontsize=14, fontweight='bold')
ax.set_xlabel('x position (mts)', fontsize=12)
ax.set_ylabel('y position (mts)', fontsize=12)
cbar = plt.colorbar(im, ax=ax)
cbar.set_label('health status\n(0=S, 1=E, 2=I, 3=R)', rotation=270, labelpad=20)

plt.tight_layout()
plt.savefig('figures/4_spatial_distribution.png', dpi=300, bbox_inches='tight')
print("saved: figures/4_spatial_distribution.png")
plt.close()

[4/8]creating spatial distribution
saved: figures/4_spatial_distribution.png


In [17]:
print("[5/8]creating temperature trajectories")

fig, ax = plt.subplots(figsize=(14, 6))

# Plot 5 random agents
sample_agents = np.random.choice(df['agent_id'].unique(), 5, replace=False)

for agent_id in sample_agents:
    agent_data = df[df['agent_id'] == agent_id].sort_values('timestep')
    ax.plot(agent_data['timestep'], agent_data['body_temperature'],
           alpha=0.7, linewidth=2, label=f'Agent {agent_id}')

ax.axhline(y=38.5, color='green', linestyle='--', alpha=0.5, label='normal (38.5°C)')
ax.axhline(y=39.5, color='orange', linestyle='--', alpha=0.5, label='fever threshold (39.5°C)')
ax.axhline(y=40.5, color='red', linestyle='--', alpha=0.5, label='high fever (40.5°C)')

ax.set_xlabel('Timestep', fontsize=12)
ax.set_ylabel('body temperature (°C)', fontsize=12)
ax.set_title('ndividual agent Ttemperature trajectories', fontsize=14, fontweight='bold')
ax.legend(loc='upper right', fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('figures/5_temperature_trajectories.png', dpi=300, bbox_inches='tight')
print("saved: figures/5_temperature_trajectories.png")
plt.close()

[5/8]creating temperature trajectories
saved: figures/5_temperature_trajectories.png


In [18]:
print("[6/8]creating breed comparison")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Infection rates by breed
ax = axes[0]
breed_infection = df.groupby(['breed', 'health_state']).size().unstack(fill_value=0)
breed_infection_pct = breed_infection.div(breed_infection.sum(axis=1), axis=0) * 100
breed_infection_pct[['EXPOSED', 'INFECTIOUS', 'RECOVERED']].plot(
    kind='bar', stacked=True, ax=ax, alpha=0.8,
    color=['yellow', 'red', 'blue']
)
ax.set_title('a)infection rates by breed', fontsize=12, fontweight='bold')
ax.set_xlabel('breed', fontsize=10)
ax.set_ylabel('percentage (%)', fontsize=10)
ax.legend(title='health state', fontsize=9)
plt.sca(ax)
plt.xticks(rotation=45, ha='right')

# Temperature by breed
ax = axes[1]
df.boxplot(column='body_temperature', by='breed', ax=ax)
ax.set_title('b)temperature distribution by breed', fontsize=12, fontweight='bold')
ax.set_xlabel('breed', fontsize=10)
ax.set_ylabel('temperature (°C)', fontsize=10)
plt.sca(ax)
plt.xticks(rotation=45, ha='right')

# Activity by breed
ax = axes[2]
df.boxplot(column='activity_level', by='breed', ax=ax)
ax.set_title('c)activity level by breed', fontsize=12, fontweight='bold')
ax.set_xlabel('breed', fontsize=10)
ax.set_ylabel('activity score', fontsize=10)
plt.sca(ax)
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.savefig('figures/6_beed_comparison.png', dpi=300, bbox_inches='tight')
print("saved: figures/6_breed_comparison.png")
plt.close()

[6/8]creating breed comparison
saved: figures/6_breed_comparison.png


In [19]:
print("[7/8]creating time series decomposition")

fig, axes = plt.subplots(4, 1, figsize=(14, 10))
fig.suptitle('fpidemic time series decomposition', fontsize=16, fontweight='bold')

# Original series
ax = axes[0]
total_infected = state_counts[['EXPOSED', 'INFECTIOUS']].sum(axis=1)
ax.plot(total_infected.index, total_infected.values, linewidth=2, color='darkred')
ax.set_ylabel('infected count', fontsize=10)
ax.set_title('a)original time series', fontsize=12)
ax.grid(True, alpha=0.3)

# 7-day moving average
ax = axes[1]
ma_7 = total_infected.rolling(window=28).mean()  # 28 timesteps = 7 hours
ax.plot(total_infected.index, ma_7.values, linewidth=2, color='blue')
ax.set_ylabel('7-Hour MA', fontsize=10)
ax.set_title('b)trend (7-hr moving average)', fontsize=12)
ax.grid(True, alpha=0.3)

# Residuals
ax = axes[2]
residuals = total_infected - ma_7
ax.plot(residuals.index, residuals.values, linewidth=1, color='green', alpha=0.7)
ax.axhline(y=0, color='black', linestyle='--', linewidth=0.8)
ax.set_ylabel('residuals', fontsize=10)
ax.set_title('c)residuals (noise)', fontsize=12)
ax.grid(True, alpha=0.3)

# ACF-like plot (simplified)
ax = axes[3]
lags = 50
acf_values = [total_infected.autocorr(lag=i) for i in range(1, lags+1)]
ax.bar(range(1, lags+1), acf_values, alpha=0.7, color='purple')
ax.axhline(y=0, color='black', linewidth=0.8)
ax.set_xlabel('lag (timesteps)', fontsize=10)
ax.set_ylabel('autocorrelation', fontsize=10)
ax.set_title('d)autocorrelation function', fontsize=12)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('figures/7_time_series_decomposition.png', dpi=300, bbox_inches='tight')
print("saved: figures/7_time_series_decomposition.png")
plt.close()

[7/8]creating time series decomposition
saved: figures/7_time_series_decomposition.png


In [20]:
print("[8/8]creating feature analysis")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('feature analysis for disease detection', fontsize=16, fontweight='bold')

# Subplot A: Temperature vs Activity scatter
ax = axes[0, 0]
for state in df['health_state'].unique():
    state_data = df[df['health_state'] == state].sample(min(500, len(df[df['health_state'] == state])))
    colors_map = {'SUSCEPTIBLE': 'green', 'EXPOSED': 'yellow',
                 'INFECTIOUS': 'red', 'RECOVERED': 'blue'}
    ax.scatter(state_data['body_temperature'], state_data['activity_level'],
              c=colors_map[state], label=state, alpha=0.4, s=20)
ax.set_xlabel('body temperature (°C)', fontsize=10)
ax.set_ylabel('activity level', fontsize=10)
ax.set_title('a)temperature vs activity', fontsize=12, fontweight='bold')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Subplot B: Cough count distribution
ax = axes[0, 1]
for state in ['SUSCEPTIBLE', 'EXPOSED', 'INFECTIOUS', 'RECOVERED']:
    state_data = df[df['health_state'] == state]['cough_count']
    ax.hist(state_data, bins=15, alpha=0.5, label=state)
ax.set_xlabel('cough count', fontsize=10)
ax.set_ylabel('frequency', fontsize=10)
ax.set_title('b)cough Count Distribution', fontsize=12, fontweight='bold')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3, axis='y')

# Subplot C: Age distribution by health state
ax = axes[1, 0]
df.boxplot(column='age_months', by='health_state', ax=ax)
ax.set_xlabel('health state', fontsize=10)
ax.set_ylabel('age (mnths)', fontsize=10)
ax.set_title('c)age distribution by health state', fontsize=12, fontweight='bold')
plt.sca(ax)
plt.xticks(rotation=45, ha='right')

# Subplot D: Summary statistics table
ax = axes[1, 1]
ax.axis('off')

summary_stats = []
for state in ['SUSCEPTIBLE', 'EXPOSED', 'INFECTIOUS', 'RECOVERED']:
    state_data = df[df['health_state'] == state]
    summary_stats.append([
        state,
        f"{state_data['body_temperature'].mean():.2f}",
        f"{state_data['activity_level'].mean():.1f}",
        f"{state_data['cough_count'].mean():.2f}",
        f"{len(state_data):,}"
    ])

table = ax.table(cellText=summary_stats,
                colLabels=['State', 'Temp (°C)', 'Activity', 'Coughs', 'N'],
                cellLoc='center',
                loc='center',
                bbox=[0, 0, 1, 1])
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1, 2)
ax.set_title('d)summary statistics by health state', fontsize=12, fontweight='bold', pad=20)

plt.tight_layout()
plt.savefig('figures/8_feature_analysis.png', dpi=300, bbox_inches='tight')
print("saved: figures/8_feature_analysis.png")
plt.close()

[8/8]creating feature analysis
saved: figures/8_feature_analysis.png


##statistical validation

In [21]:
# Temperature normality tests
print("\ntemperature distribution tests:")
for state in df['health_state'].unique():
    temps = df[df['health_state'] == state]['body_temperature']
    if len(temps) > 5000:
        temps = temps.sample(5000)
    statistic, p_value = stats.shapiro(temps)
    print(f" {state:12s}: mean={temps.mean():.2f}°C, std={temps.std():.2f}°C, "
          f"normality p={p_value:.4f}")

# Data quality metrics
print("\ndata quality metrics:")
print(f"total records: {len(df):,}")
print(f"missing values: {df.isnull().sum().sum()}")
print(f"duplicate rows: {df.duplicated().sum()}")
print(f"features: {len(df.columns)}")
print(f"temporal completeness: 100%")

# Feature statistics
print("\nfeature statistics:")
for col in ['body_temperature', 'activity_level', 'cough_count']:
    print(f"{col}:")
    print(f"range: [{df[col].min():.2f}, {df[col].max():.2f}]")
    print(f"mean: {df[col].mean():.2f}, std: {df[col].std():.2f}")

# Correlation analysis
print("\nkey corr:")
high_corr = []
for i in range(len(numeric_cols)):
    for j in range(i+1, len(numeric_cols)):
        corr = df[numeric_cols[i]].corr(df[numeric_cols[j]])
        if abs(corr) > 0.5:
            high_corr.append((numeric_cols[i], numeric_cols[j], corr))

high_corr.sort(key=lambda x: abs(x[2]), reverse=True)
for feat1, feat2, corr in high_corr[:5]:
    print(f" {feat1:20s} ↔ {feat2:20s}: {corr:+.3f}")


temperature distribution tests:
 EXPOSED     : mean=38.89°C, std=0.27°C, normality p=0.0000
 SUSCEPTIBLE : mean=38.48°C, std=0.27°C, normality p=0.0000
 INFECTIOUS  : mean=40.77°C, std=0.40°C, normality p=0.0000
 RECOVERED   : mean=38.51°C, std=0.30°C, normality p=0.0000

data quality metrics:
total records: 100,000
missing values: 0
duplicate rows: 0
features: 19
temporal completeness: 100%

feature statistics:
body_temperature:
range: [37.56, 41.90]
mean: 38.68, std: 0.68
activity_level:
range: [0.00, 170.70]
mean: 81.33, std: 47.34
cough_count:
range: [0.00, 15.00]
mean: 0.77, std: 1.60

key corr:
 body_temperature     ↔ heavy_breathing_count: +0.804
 cough_count          ↔ heavy_breathing_count: +0.742
 body_temperature     ↔ sneeze_count        : +0.727
 sneeze_count         ↔ heavy_breathing_count: +0.707
 body_temperature     ↔ cough_count         : +0.705


In [22]:
os.makedirs('models', exist_ok=True)
os.makedirs('figures', exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [23]:
class HerdHealthDataset(Dataset):
    def __init__(self, df, temporal_window=12, forecast_horizon=4,
                 proximity_threshold=5.0):
        self.df = df.sort_values(['timestep', 'agent_id']).reset_index(drop=True)
        self.temporal_window = temporal_window
        self.forecast_horizon = forecast_horizon
        self.proximity_threshold = proximity_threshold

        # Get unique timesteps and agents
        self.timesteps = sorted(df['timestep'].unique())
        self.agents = sorted(df['agent_id'].unique())
        self.num_agents = len(self.agents)

        # Feature columns
        self.feature_cols = [
            'body_temperature', 'activity_level', 'cough_count',
            'sneeze_count', 'heavy_breathing_count', 'age_months',
            'susceptibility'
        ]
        self.num_features = len(self.feature_cols)

        # Valid timesteps for sampling (need history + future)
        self.valid_timesteps = [
            t for t in self.timesteps
            if t >= temporal_window and t < max(self.timesteps) - forecast_horizon
        ]

        print(f"dataset initialized")
        print(f"agents: {self.num_agents}")
        print(f"valid timesteps: {len(self.valid_timesteps)}")
        print(f"samples: {len(self.valid_timesteps)}")
        print(f"features per node: {self.num_features}")

    def __len__(self):
        return len(self.valid_timesteps)

    def __getitem__(self, idx):
        current_t = self.valid_timesteps[idx]

        # Get temporal window of data [t-window:t]
        temporal_data = []
        for t in range(current_t - self.temporal_window, current_t):
            snapshot = self.df[self.df['timestep'] == t]
            snapshot = snapshot.sort_values('agent_id')

            # Extract features
            features = snapshot[self.feature_cols].values
            temporal_data.append(features)

        # Stack: [temporal_window, num_agents, num_features]
        temporal_data = np.stack(temporal_data, axis=0)

        # Transpose to [num_agents, temporal_window, num_features]
        node_features = np.transpose(temporal_data, (1, 0, 2))

        # Get positions at current time for adjacency matrix
        current_snapshot = self.df[self.df['timestep'] == current_t].sort_values('agent_id')
        positions = current_snapshot[['position_x', 'position_y']].values

        # Build adjacency matrix (proximity-based)
        distances = cdist(positions, positions, metric='euclidean')
        adj_matrix = (distances < self.proximity_threshold).astype(np.float32)
        np.fill_diagonal(adj_matrix, 0)  # Remove self-loops

        # Get target labels at t+forecast_horizon
        target_t = current_t + self.forecast_horizon
        target_snapshot = self.df[self.df['timestep'] == target_t].sort_values('agent_id')

        # Binary classification: Infected (E, I, R) vs Healthy (S)
        labels = (target_snapshot['health_state'] != 'SUSCEPTIBLE').astype(int).values

        return (
            torch.FloatTensor(node_features),
            torch.FloatTensor(adj_matrix),
            torch.LongTensor(labels)
        )

In [24]:
class LearnableGraphConv(nn.Module):
    def __init__(self, in_channels, out_channels, num_nodes):
        super(LearnableGraphConv, self).__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.num_nodes = num_nodes

        # Learnable Laplacian components
        self.alpha = nn.Parameter(torch.randn(num_nodes, num_nodes) * 0.01)

        # Feature transformation
        self.weight = nn.Parameter(torch.randn(in_channels, out_channels))
        self.bias = nn.Parameter(torch.zeros(out_channels))

        # Initialize
        nn.init.xavier_uniform_(self.weight)

    def forward(self, x, adj):
        batch_size = x.size(0)

        # Ensure adj has batch dimension
        if adj.dim() == 2:
            adj = adj.unsqueeze(0).repeat(batch_size, 1, 1)

        # Compute learnable Laplacian
        # Symmetrize alpha
        alpha_sym = (self.alpha + self.alpha.t()) / 2

        # Combine with input adjacency
        # L = I - D^(-1/2) * (A + alpha) * D^(-1/2)
        A_refined = adj + torch.sigmoid(alpha_sym).unsqueeze(0)

        # Degree matrix
        D = torch.sum(A_refined, dim=-1)
        D_inv_sqrt = torch.pow(D + 1e-6, -0.5)
        D_inv_sqrt = torch.diag_embed(D_inv_sqrt)

        # Normalized adjacency
        A_norm = torch.bmm(torch.bmm(D_inv_sqrt, A_refined), D_inv_sqrt)

        # Graph convolution: A_norm @ X @ W
        out = torch.bmm(A_norm, x)  # [batch, num_nodes, in_channels]
        out = torch.matmul(out, self.weight)  # [batch, num_nodes, out_channels]
        out = out + self.bias

        return out

class TemporalConv1D(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3):
        super(TemporalConv1D, self).__init__()
        self.conv = nn.Conv1d(
            in_channels,
            out_channels,
            kernel_size=kernel_size,
            padding=kernel_size//2
        )
        self.bn = nn.BatchNorm1d(out_channels)

    def forward(self, x):
        batch, nodes, temporal, channels = x.size()

        # Reshape for 1D conv: [batch*nodes, channels, temporal]
        x = x.permute(0, 1, 3, 2).reshape(batch * nodes, channels, temporal)

        # Conv
        out = self.conv(x)
        out = self.bn(out)
        out = F.relu(out)

        # Reshape back: [batch, nodes, temporal, out_channels]
        out = out.reshape(batch, nodes, -1, temporal).permute(0, 1, 3, 2)

        return out

In [25]:
class STGNNBlock(nn.Module):
    def __init__(self, in_channels, spatial_channels, out_channels, num_nodes, temporal_kernel=3):
        super(STGNNBlock, self).__init__()

        # Temporal modeling
        self.temporal1 = TemporalConv1D(in_channels, spatial_channels,
                                        kernel_size=temporal_kernel)

        # Spatial modeling (graph convolution)
        self.graph_conv = LearnableGraphConv(spatial_channels, spatial_channels,
                                            num_nodes)

        # Temporal modeling
        self.temporal2 = TemporalConv1D(spatial_channels, out_channels,
                                        kernel_size=temporal_kernel)

        # Residual connection
        self.residual = nn.Conv1d(in_channels, out_channels, kernel_size=1) \
                        if in_channels != out_channels else None

        self.dropout = nn.Dropout(0.3)

    def forward(self, x, adj):
        residual = x

        # Temporal conv 1
        out = self.temporal1(x)  # [batch, nodes, temporal, spatial_channels]

        # Graph conv (apply at each time step)
        batch, nodes, temporal, channels = out.size()
        out_gcn = []
        for t in range(temporal):
            x_t = out[:, :, t, :]  # [batch, nodes, channels]
            out_t = self.graph_conv(x_t, adj)  # [batch, nodes, spatial_channels]
            out_gcn.append(out_t.unsqueeze(2))
        out = torch.cat(out_gcn, dim=2)  # [batch, nodes, temporal, spatial_channels]

        # Temporal conv 2
        out = self.temporal2(out)  # [batch, nodes, temporal, out_channels]

        # Residual connection
        if self.residual is not None:
            batch, nodes, temporal, in_ch = residual.size()
            residual = residual.permute(0, 1, 3, 2).reshape(batch * nodes, in_ch, temporal)
            residual = self.residual(residual)
            residual = residual.reshape(batch, nodes, -1, temporal).permute(0, 1, 3, 2)

        out = F.relu(out + residual)
        out = self.dropout(out)

        return out

class HerdHealthSTGNN(nn.Module):
    def __init__(self, num_features, num_nodes, temporal_window,
                 hidden_channels=64, num_classes=2):
        super(HerdHealthSTGNN, self).__init__()

        self.num_nodes = num_nodes
        self.temporal_window = temporal_window

        # Input projection
        self.input_proj = nn.Linear(num_features, hidden_channels)

        # STGNN blocks
        self.block1 = STGNNBlock(hidden_channels, hidden_channels,
                                hidden_channels, num_nodes)
        self.block2 = STGNNBlock(hidden_channels, hidden_channels,
                                hidden_channels, num_nodes)
        self.block3 = STGNNBlock(hidden_channels, hidden_channels,
                                hidden_channels, num_nodes)

        # Temporal pooling (attention-based)
        self.temporal_attention = nn.Linear(hidden_channels, 1)

        # Output classifier
        self.classifier = nn.Sequential(
            nn.Linear(hidden_channels, hidden_channels // 2),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_channels // 2, num_classes)
        )

    def forward(self, x, adj):
        # Input projection
        batch, nodes, temporal, features = x.size()
        x = self.input_proj(x)  # [batch, nodes, temporal, hidden]

        # STGNN blocks
        x = self.block1(x, adj)
        x = self.block2(x, adj)
        x = self.block3(x, adj)

        # Temporal attention pooling
        attn_scores = self.temporal_attention(x)  # [batch, nodes, temporal, 1]
        attn_weights = F.softmax(attn_scores, dim=2)
        x = torch.sum(x * attn_weights, dim=2)  # [batch, nodes, hidden]

        # Classification
        out = self.classifier(x)  # [batch, nodes, num_classes]

        return out

In [26]:
def train_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    all_preds = []
    all_labels = []

    for node_features, adj, labels in tqdm(dataloader, desc="Training", leave=False):
        node_features = node_features.to(device)
        adj = adj.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        # Forward
        outputs = model(node_features, adj)  # [batch, nodes, num_classes]

        # Reshape for loss
        batch, nodes, num_classes = outputs.size()
        outputs = outputs.reshape(-1, num_classes)
        labels = labels.reshape(-1)

        loss = criterion(outputs, labels)

        # Backward
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()

        # Predictions
        preds = torch.argmax(outputs, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(dataloader)
    accuracy = accuracy_score(all_labels, all_preds)

    return avg_loss, accuracy

def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds = []
    all_probs = []
    all_labels = []

    with torch.no_grad():
        for node_features, adj, labels in tqdm(dataloader, desc="Evaluating", leave=False):
            node_features = node_features.to(device)
            adj = adj.to(device)
            labels = labels.to(device)

            # Forward
            outputs = model(node_features, adj)

            # Reshape
            batch, nodes, num_classes = outputs.size()
            outputs = outputs.reshape(-1, num_classes)
            labels = labels.reshape(-1)

            loss = criterion(outputs, labels)
            total_loss += loss.item()

            # Predictions
            probs = F.softmax(outputs, dim=1)
            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs[:, 1].cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(dataloader)
    accuracy = accuracy_score(all_labels, all_preds)

    try:
        roc_auc = roc_auc_score(all_labels, all_probs)
    except:
        roc_auc = 0.0

    return avg_loss, accuracy, roc_auc, all_preds, all_probs, all_labels

In [27]:
print("loading dataset")
df = pd.read_csv('data/digital_herd_health_dataset.csv')
print(f"loaded {len(df):,} records\n")

# Create dataset
print("creating spatiotemporal dataset")
dataset = HerdHealthDataset(
    df,
    temporal_window=12,
    forecast_horizon=4,
    proximity_threshold=5.0
)

# Train/Val split (80/20)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(
    dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

print(f"\ntrain samples: {len(train_dataset)}")
print(f"val samples: {len(val_dataset)}\n")

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=0)

# Initialize model
print("\ninitializing STGNN model")
model = HerdHealthSTGNN(
    num_features=dataset.num_features,
    num_nodes=dataset.num_agents,
    temporal_window=dataset.temporal_window,
    hidden_channels=64,
    num_classes=2
).to(device)

print(f"\nmodel parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"\nmodel architecture:")
print(model)
print()

# Training setup
criterion = nn.CrossEntropyLoss()
optimizer = Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

# Training loop
print("TRAINING SPATIOTEMPORAL GNN")

num_epochs = 20
best_val_acc = 0.0
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'val_roc_auc': []}

for epoch in range(num_epochs):
    print(f"\nepoch {epoch+1}/{num_epochs}")

    # Train
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device)

    # Validate
    val_loss, val_acc, val_roc_auc, val_preds, val_probs, val_labels = evaluate(
        model, val_loader, criterion, device
    )

    # Scheduler step
    scheduler.step(val_loss)

    # Log
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['val_roc_auc'].append(val_roc_auc)

    print(f"train loss: {train_loss:.4f} | train acc: {train_acc:.4f}")
    print(f"val loss: {val_loss:.4f} | val acc: {val_acc:.4f} | val ROC-AUC: {val_roc_auc:.4f}")

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_acc,
            'val_roc_auc': val_roc_auc
        }, 'models/stgnn_best.pth')
        print(f" best model saved (val acc: {best_val_acc:.4f})")

    # Early stopping
    if epoch > 20 and val_acc < 0.92:
        print("\ntraining seems stuck. consider adjusting hyperparameters.")

    if epoch > 30 and best_val_acc > 0.97:
        print(f"\ntarget accuracy achieved! best val acc: {best_val_acc:.4f}")
        break

print(f"best validation accuracy: {best_val_acc:.4f}")

loading dataset
loaded 100,000 records

creating spatiotemporal dataset
dataset initialized
agents: 100
valid timesteps: 983
samples: 983
features per node: 7

train samples: 786
val samples: 197


initializing STGNN model

model parameters: 120,083

model architecture:
HerdHealthSTGNN(
  (input_proj): Linear(in_features=7, out_features=64, bias=True)
  (block1): STGNNBlock(
    (temporal1): TemporalConv1D(
      (conv): Conv1d(64, 64, kernel_size=(3,), stride=(1,), padding=(1,))
      (bn): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (graph_conv): LearnableGraphConv()
    (temporal2): TemporalConv1D(
      (conv): Conv1d(64, 64, kernel_size=(3,), stride=(1,), padding=(1,))
      (bn): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (dropout): Dropout(p=0.3, inplace=False)
  )
  (block2): STGNNBlock(
    (temporal1): TemporalConv1D(
      (conv): Conv1d(64, 64, kernel_size=(3,), stride=(1,), padding

train loss: 1.7486 | train acc: 0.5359
val loss: 0.6599 | val acc: 0.6745 | val ROC-AUC: 0.6847
 best model saved (val acc: 0.6745)

epoch 2/20


train loss: 0.6331 | train acc: 0.6594
val loss: 0.4774 | val acc: 0.7593 | val ROC-AUC: 0.8858
 best model saved (val acc: 0.7593)

epoch 3/20


train loss: 0.4296 | train acc: 0.8113
val loss: 0.2788 | val acc: 0.8929 | val ROC-AUC: 0.9524
 best model saved (val acc: 0.8929)

epoch 4/20


train loss: 0.3507 | train acc: 0.8672
val loss: 0.2372 | val acc: 0.9055 | val ROC-AUC: 0.9676
 best model saved (val acc: 0.9055)

epoch 5/20


...

train loss: 0.0929 | train acc: 0.9742
val loss: 0.2626 | val acc: 0.8964 | val ROC-AUC: 0.9944

epoch 17/20


train loss: 0.0916 | train acc: 0.9747
val loss: 0.2182 | val acc: 0.9190 | val ROC-AUC: 0.9941

epoch 18/20


train loss: 0.0895 | train acc: 0.9766
val loss: 0.3266 | val acc: 0.8911 | val ROC-AUC: 0.9921

epoch 19/20


train loss: 0.0891 | train acc: 0.9760
val loss: 0.1915 | val acc: 0.9335 | val ROC-AUC: 0.9957
 best model saved (val acc: 0.9335)

epoch 20/20


train loss: 0.0845 | train acc: 0.9763
val loss: 0.2763 | val acc: 0.9015 | val ROC-AUC: 0.9913
best validation accuracy: 0.9335


In [28]:
os.makedirs('figures', exist_ok=True)

print("loading dataset")
df = pd.read_csv('data/digital_herd_health_dataset.csv')
print(f"loaded {len(df):,} records\n")

animation_steps = 290
df_anim = df[df['timestep'] < animation_steps].copy()

timesteps = sorted(df_anim['timestep'].unique())
agents = sorted(df_anim['agent_id'].unique())
num_agents = len(agents)

print(f"animation parameters:")
print(f"timesteps: {len(timesteps)}")
print(f"agents: {num_agents}")
print(f"duration: {len(timesteps)/4:.1f} hrs\n")

gnn_predictions = None
try:
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = HerdHealthSTGNN(
        num_features=7,
        num_nodes=num_agents,
        temporal_window=12,
        hidden_channels=64,
        num_classes=2
    ).to(device)

    checkpoint = torch.load('models/stgnn_best.pth', map_location=device, weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    print("GNN model loaded\n")
except Exception as e:
    print(f"Error loading GNN model: {e}. Animation will show ground truth only.\n")
    import traceback
    traceback.print_exc()

loading dataset
loaded 100,000 records

animation parameters:
timesteps: 290
agents: 100
duration: 72.5 hrs

GNN model loaded



In [29]:
colors = {
    'SUSCEPTIBLE': '#2ecc71',  # Green
    'EXPOSED': '#f39c12',       # Orange
    'INFECTIOUS': '#e74c3c',    # Red
    'RECOVERED': '#3498db'      # Blue
}

animation_data = []
for t in timesteps:
    snapshot = df_anim[df_anim['timestep'] == t].sort_values('agent_id')
    animation_data.append({
        'timestep': t,
        'positions': snapshot[['position_x', 'position_y']].values,
        'health_states': snapshot['health_state'].values,
        'temperatures': snapshot['body_temperature'].values,
        'activities': snapshot['activity_level'].values
    })

print(f"prepared {len(animation_data)} frames\n")

fig, axes = plt.subplots(2, 2, figsize=(16, 16))
fig.suptitle('Digital Herd Health Twin - Spatiotemporal Disease Dynamics',
            fontsize=18, fontweight='bold')

# Initialize plots
ax_spatial = axes[0, 0]  # Spatial distribution
ax_network = axes[0, 1]  # Contact network
ax_metrics = axes[1, 0]  # Time series metrics
ax_temp = axes[1, 1]     # Temperature distribution

# Metrics tracking
metrics_data = {
    'susceptible': [],
    'exposed': [],
    'infectious': [],
    'recovered': []
}

def init():
    for ax in axes.flat:
        ax.clear()
    return []

def animate(frame_idx):
    for ax in axes.flat:
        ax.clear()

    data = animation_data[frame_idx]
    t = data['timestep']
    positions = data['positions']
    states = data['health_states']
    temps = data['temperatures']
    activities = data['activities']

    ax = ax_spatial
    ax.set_xlim(0, 50)
    ax.set_ylim(0, 50)
    ax.set_aspect('equal')
    ax.set_title(f'A) Spatial Distribution (t={t}, hour={t/4:.1f}h)',
                fontsize=12, fontweight='bold')
    ax.set_xlabel('X Position (meters)')
    ax.set_ylabel('Y Position (meters)')

    # Draw farm zones (feeding stations, water troughs)
    # Feeding stations
    feeding_stations = [(12, 12), (38, 12), (12, 38), (38, 38)]
    for fx, fy in feeding_stations:
        rect = Rectangle((fx-2.5, fy-2.5), 5, 5,
                        facecolor='wheat', edgecolor='brown',
                        alpha=0.3, linewidth=2)
        ax.add_patch(rect)

    # Water troughs
    water_troughs = [(25, 12), (25, 38)]
    for wx, wy in water_troughs:
        circle = Circle((wx, wy), 2, facecolor='lightblue',
                       edgecolor='blue', alpha=0.4, linewidth=2)
        ax.add_patch(circle)

    # Plot agents
    for i, (pos, state) in enumerate(zip(positions, states)):
        color = colors[state]
        circle = Circle(pos, 0.8, facecolor=color, edgecolor='black',
                       linewidth=1.5, alpha=0.8)
        ax.add_patch(circle)

    # Legend
    legend_elements = [
        Circle((0, 0), 0.5, facecolor=colors['SUSCEPTIBLE'],
               edgecolor='black', label='Susceptible'),
        Circle((0, 0), 0.5, facecolor=colors['EXPOSED'],
               edgecolor='black', label='Exposed'),
        Circle((0, 0), 0.5, facecolor=colors['INFECTIOUS'],
               edgecolor='black', label='Infectious'),
        Circle((0, 0), 0.5, facecolor=colors['RECOVERED'],
               edgecolor='black', label='Recovered')
    ]
    ax.legend(handles=legend_elements, loc='upper right', fontsize=9)
    ax.grid(True, alpha=0.2)

    ax = ax_network
    ax.set_xlim(0, 50)
    ax.set_ylim(0, 50)
    ax.set_aspect('equal')
    ax.set_title('B) Contact Network (5m proximity)',
                fontsize=12, fontweight='bold')
    ax.set_xlabel('X Position (meters)')
    ax.set_ylabel('Y Position (meters)')

    # Compute distances
    distances = cdist(positions, positions, metric='euclidean')
    proximity_threshold = 5.0

    # Draw edges (contacts)
    for i in range(len(positions)):
        for j in range(i+1, len(positions)):
            if distances[i, j] < proximity_threshold:
                # Draw edge
                x_vals = [positions[i, 0], positions[j, 0]]
                y_vals = [positions[i, 1], positions[j, 1]]

                # Color edge by interaction type
                if states[i] == 'INFECTIOUS' or states[j] == 'INFECTIOUS':
                    edge_color = 'red'
                    edge_alpha = 0.6
                    edge_width = 2
                else:
                    edge_color = 'gray'
                    edge_alpha = 0.2
                    edge_width = 1

                ax.plot(x_vals, y_vals, color=edge_color,
                       alpha=edge_alpha, linewidth=edge_width)

    # Draw nodes
    for pos, state in zip(positions, states):
        color = colors[state]
        circle = Circle(pos, 0.8, facecolor=color, edgecolor='black',
                       linewidth=1.5, alpha=0.9)
        ax.add_patch(circle)

    # Count contacts
    num_contacts = np.sum(distances < proximity_threshold) // 2
    infectious_count = np.sum(states == 'INFECTIOUS')
    ax.text(0.02, 0.98, f'Active Contacts: {num_contacts}\nInfectious Agents: {infectious_count}',
           transform=ax.transAxes, verticalalignment='top',
           bbox=dict(boxstyle='round', facecolor='white', alpha=0.8),
           fontsize=10)

    ax.grid(True, alpha=0.2)

    ax = ax_metrics

    # Update metrics
    state_counts = pd.Series(states).value_counts()
    metrics_data['susceptible'].append(state_counts.get('SUSCEPTIBLE', 0))
    metrics_data['exposed'].append(state_counts.get('EXPOSED', 0))
    metrics_data['infectious'].append(state_counts.get('INFECTIOUS', 0))
    metrics_data['recovered'].append(state_counts.get('RECOVERED', 0))

    # Plot curves
    time_axis = list(range(len(metrics_data['susceptible'])))

    ax.plot(time_axis, metrics_data['susceptible'],
           color=colors['SUSCEPTIBLE'], linewidth=2.5, label='Susceptible')
    ax.plot(time_axis, metrics_data['exposed'],
           color=colors['EXPOSED'], linewidth=2.5, label='Exposed')
    ax.plot(time_axis, metrics_data['infectious'],
           color=colors['INFECTIOUS'], linewidth=2.5, label='Infectious')
    ax.plot(time_axis, metrics_data['recovered'],
           color=colors['RECOVERED'], linewidth=2.5, label='Recovered')

    ax.set_xlabel('Timestep', fontsize=10)
    ax.set_ylabel('Number of Animals', fontsize=10)
    ax.set_title('C) SEIRS Epidemic Dynamics', fontsize=12, fontweight='bold')
    ax.legend(loc='best', fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(0, len(timesteps))
    ax.set_ylim(0, num_agents + 5)

    # Mark current time
    ax.axvline(x=frame_idx, color='black', linestyle='--', alpha=0.5)
    ax = ax_temp

    # Group temperatures by state
    temp_by_state = {}
    for state in ['SUSCEPTIBLE', 'EXPOSED', 'INFECTIOUS', 'RECOVERED']:
        mask = states == state
        if np.any(mask):
            temp_by_state[state] = temps[mask]

    # Violin plot
    data_to_plot = []
    labels = []
    colors_list = []

    for state in ['SUSCEPTIBLE', 'EXPOSED', 'INFECTIOUS', 'RECOVERED']:
        if state in temp_by_state and len(temp_by_state[state]) > 0:
            data_to_plot.append(temp_by_state[state])
            labels.append(state)
            colors_list.append(colors[state])

    if data_to_plot:
        parts = ax.violinplot(data_to_plot, positions=range(len(data_to_plot)),
                             showmeans=True, showmedians=True)

        # Color the violins
        for i, pc in enumerate(parts['bodies']):
            pc.set_facecolor(colors_list[i])
            pc.set_alpha(0.7)

    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha='right')
    ax.set_ylabel('Body Temperature (°C)', fontsize=10)
    ax.set_title('D) Temperature Distribution', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    ax.set_ylim(37.5, 42)

    # Add threshold lines
    ax.axhline(y=38.5, color='green', linestyle='--', alpha=0.5, label='Normal')
    ax.axhline(y=39.5, color='orange', linestyle='--', alpha=0.5, label='Fever')
    ax.axhline(y=40.5, color='red', linestyle='--', alpha=0.5, label='High Fever')
    ax.legend(loc='upper right', fontsize=8)

    plt.tight_layout()

    return []


print("rendering animation frames")

anim = animation.FuncAnimation(
    fig,
    animate,
    init_func=init,
    frames=len(animation_data),
    interval=100,
    blit=False,
    repeat=True
)

# Save as GIF
output_path = 'figures/9_simulation_animation.gif'

# Use pillow writer for compatibility
from matplotlib.animation import PillowWriter
writer = PillowWriter(fps=10)

anim.save(output_path, writer=writer, dpi=80)

print(f"animation saved: {output_path}")
print(f"file size: {os.path.getsize(output_path) / 1024 / 1024:.2f} MB")

# Also save a static frame at peak infection
# Determine the peak infection frame from the full metrics_log DataFrame
peak_frame = np.argmax(metrics_log['infectious'])
print(f"\nsaving snapshot at peak infection (frame {peak_frame})")

fig_static, axes_static = plt.subplots(2, 2, figsize=(16, 16))
fig_static.suptitle(f'Disease Spread Snapshot - Peak Infection (Hour {peak_frame/4:.1f})',
                   fontsize=18, fontweight='bold')

# Re-render the peak frame (make sure this frame is within the animation_steps range)
if peak_frame < animation_steps:
    animate(peak_frame)
    plt.savefig('figures/10_peak_infection_snapshot.png', dpi=300, bbox_inches='tight')
    print("snapshot saved: figures/10_peak_infection_snapshot.png")
else:
    print(f"Peak infection frame ({peak_frame}) is outside the animation range ({animation_steps}). Skipping snapshot.")

plt.close('all')

prepared 290 frames

rendering animation frames
animation saved: figures/9_simulation_animation.gif
file size: 27.36 MB

saving snapshot at peak infection (frame 282)
snapshot saved: figures/10_peak_infection_snapshot.png


In [30]:
peak_frame = np.argmax(metrics_log['infectious'])
print(f"\nsaving snapshot at peak infection (frame {peak_frame})")

fig_static, axes_static = plt.subplots(2, 2, figsize=(16, 16))
fig_static.suptitle(f'Disease Spread Snapshot - Peak Infection (Hour {peak_frame/4:.1f})',
                   fontsize=18, fontweight='bold')

# --- Start of modified code for static snapshot ---
# Get data for the peak frame directly from the full dataframe
peak_timestep = peak_frame # Use the peak frame index as the timestep
peak_data_full_df = df[df['timestep'] == peak_timestep].sort_values('agent_id')

if not peak_data_full_df.empty:
    t_peak = peak_data_full_df['timestep'].iloc[0]
    positions_peak = peak_data_full_df[['position_x', 'position_y']].values
    states_peak = peak_data_full_df['health_state'].values
    temps_peak = peak_data_full_df['body_temperature'].values
    activities_peak = peak_data_full_df['activity_level'].values

    # Plot Spatial Distribution
    ax_spatial_static = axes_static[0, 0]
    ax_spatial_static.set_xlim(0, 50)
    ax_spatial_static.set_ylim(0, 50)
    ax_spatial_static.set_aspect('equal')
    ax_spatial_static.set_title(f'A) Spatial Distribution (t={t_peak}, hour={t_peak/4:.1f}h)',
                                fontsize=12, fontweight='bold')
    ax_spatial_static.set_xlabel('X Position (meters)')
    ax_spatial_static.set_ylabel('Y Position (meters)')

    # Draw farm zones
    feeding_stations = [(12, 12), (38, 12), (12, 38), (38, 38)]
    for fx, fy in feeding_stations:
        rect = Rectangle((fx-2.5, fy-2.5), 5, 5, facecolor='wheat', edgecolor='brown', alpha=0.3, linewidth=2)
        ax_spatial_static.add_patch(rect)
    water_troughs = [(25, 12), (25, 38)]
    for wx, wy in water_troughs:
        circle = Circle((wx, wy), 2, facecolor='lightblue', edgecolor='blue', alpha=0.4, linewidth=2)
        ax_spatial_static.add_patch(circle)

    # Plot agents
    for pos, state in zip(positions_peak, states_peak):
        color = colors[state]
        circle = Circle(pos, 0.8, facecolor=color, edgecolor='black', linewidth=1.5, alpha=0.8)
        ax_spatial_static.add_patch(circle)

    legend_elements = [
        Circle((0, 0), 0.5, facecolor=colors['SUSCEPTIBLE'], edgecolor='black', label='Susceptible'),
        Circle((0, 0), 0.5, facecolor=colors['EXPOSED'], edgecolor='black', label='Exposed'),
        Circle((0, 0), 0.5, facecolor=colors['INFECTIOUS'], edgecolor='black', label='Infectious'),
        Circle((0, 0), 0.5, facecolor=colors['RECOVERED'], edgecolor='black', label='Recovered')
    ]
    ax_spatial_static.legend(handles=legend_elements, loc='upper right', fontsize=9)
    ax_spatial_static.grid(True, alpha=0.2)

    # Plot Contact Network
    ax_network_static = axes_static[0, 1]
    ax_network_static.set_xlim(0, 50)
    ax_network_static.set_ylim(0, 50)
    ax_network_static.set_aspect('equal')
    ax_network_static.set_title('B) Contact Network (5m proximity)', fontsize=12, fontweight='bold')
    ax_network_static.set_xlabel('X Position (meters)')
    ax_network_static.set_ylabel('Y Position (meters)')

    distances = cdist(positions_peak, positions_peak, metric='euclidean')
    proximity_threshold = 5.0

    for i in range(len(positions_peak)):
        for j in range(i+1, len(positions_peak)):
            if distances[i, j] < proximity_threshold:
                x_vals = [positions_peak[i, 0], positions_peak[j, 0]]
                y_vals = [positions_peak[i, 1], positions_peak[j, 1]]
                if states_peak[i] == 'INFECTIOUS' or states_peak[j] == 'INFECTIOUS':
                    edge_color = 'red'
                    edge_alpha = 0.6
                    edge_width = 2
                else:
                    edge_color = 'gray'
                    edge_alpha = 0.2
                    edge_width = 1
                ax_network_static.plot(x_vals, y_vals, color=edge_color, alpha=edge_alpha, linewidth=edge_width)

    for pos, state in zip(positions_peak, states_peak):
        color = colors[state]
        circle = Circle(pos, 0.8, facecolor=color, edgecolor='black', linewidth=1.5, alpha=0.9)
        ax_network_static.add_patch(circle)

    num_contacts = np.sum(distances < proximity_threshold) // 2
    infectious_count = np.sum(states_peak == 'INFECTIOUS')
    ax_network_static.text(0.02, 0.98, f'Active Contacts: {num_contacts}\nInfectious Agents: {infectious_count}',
                           transform=ax_network_static.transAxes, verticalalignment='top',
                           bbox=dict(boxstyle='round', facecolor='white', alpha=0.8), fontsize=10)
    ax_network_static.grid(True, alpha=0.2)

    # Plot SEIRS Epidemic Dynamics (full time series)
    ax_metrics_static = axes_static[1, 0]
    time_axis = list(range(len(metrics_log['susceptible'])))
    ax_metrics_static.plot(time_axis, metrics_log['susceptible'], color=colors['SUSCEPTIBLE'], linewidth=2.5, label='Susceptible')
    ax_metrics_static.plot(time_axis, metrics_log['exposed'], color=colors['EXPOSED'], linewidth=2.5, label='Exposed')
    ax_metrics_static.plot(time_axis, metrics_log['infectious'], color=colors['INFECTIOUS'], linewidth=2.5, label='Infectious')
    ax_metrics_static.plot(time_axis, metrics_log['recovered'], color=colors['RECOVERED'], linewidth=2.5, label='Recovered')

    ax_metrics_static.set_xlabel('Timestep', fontsize=10)
    ax_metrics_static.set_ylabel('Number of Animals', fontsize=10)
    ax_metrics_static.set_title('C) SEIRS Epidemic Dynamics (Full Simulation)', fontsize=12, fontweight='bold')
    ax_metrics_static.legend(loc='best', fontsize=9)
    ax_metrics_static.grid(True, alpha=0.3)
    ax_metrics_static.set_xlim(0, len(metrics_log['susceptible']))
    ax_metrics_static.set_ylim(0, num_agents + 5)
    ax_metrics_static.axvline(x=peak_frame, color='black', linestyle='--', alpha=0.5, label='Peak Infection')
    ax_metrics_static.legend(loc='best', fontsize=9)


    # Plot Temperature Distribution
    ax_temp_static = axes_static[1, 1]
    temp_by_state_peak = {}
    for state in ['SUSCEPTIBLE', 'EXPOSED', 'INFECTIOUS', 'RECOVERED']:
        mask = states_peak == state
        if np.any(mask):
            temp_by_state_peak[state] = temps_peak[mask]

    data_to_plot_static = []
    labels_static = []
    colors_list_static = []

    for state in ['SUSCEPTIBLE', 'EXPOSED', 'INFECTIOUS', 'RECOVERED']:
         if state in temp_by_state_peak and len(temp_by_state_peak[state]) > 0:
             data_to_plot_static.append(temp_by_state_peak[state])
             labels_static.append(state)
             colors_list_static.append(colors[state])


    if data_to_plot_static:
        parts_static = ax_temp_static.violinplot(data_to_plot_static, positions=range(len(data_to_plot_static)),
                                                 showmeans=True, showmedians=True)
        for i, pc in enumerate(parts_static['bodies']):
            pc.set_facecolor(colors_list_static[i])
            pc.set_alpha(0.7)

    ax_temp_static.set_xticks(range(len(labels_static)))
    ax_temp_static.set_xticklabels(labels_static, rotation=45, ha='right')
    ax_temp_static.set_ylabel('Body Temperature (°C)', fontsize=10)
    ax_temp_static.set_title('D) Temperature Distribution', fontsize=12, fontweight='bold')
    ax_temp_static.grid(True, alpha=0.3, axis='y')
    ax_temp_static.set_ylim(37.5, 42)
    ax_temp_static.axhline(y=38.5, color='green', linestyle='--', alpha=0.5, label='Normal')
    ax_temp_static.axhline(y=39.5, color='orange', linestyle='--', alpha=0.5, label='Fever')
    ax_temp_static.axhline(y=40.5, color='red', linestyle='--', alpha=0.5, label='High Fever')
    ax_temp_static.legend(loc='upper right', fontsize=8)

    plt.tight_layout()
    plt.savefig('figures/10_peak_infection_snapshot.png', dpi=300, bbox_inches='tight')
    print("snapshot saved: figures/10_peak_infection_snapshot.png")
else:
    print(f"Data for peak infection frame ({peak_frame}) not found in the full dataset. Skipping snapshot.")

plt.close('all')


saving snapshot at peak infection (frame 282)
snapshot saved: figures/10_peak_infection_snapshot.png
